# 09 — Never-worse, label-efficient calibration of a new user (proposed method vs baselines)

**Honest correction of notebook 08:** its "session moment matching" (40.7) used the statistics of *all* 16,540 training trials of the new subject (~3.7 h of recording), and the k-label methods inherited those statistics. Here **every statistic of the new user comes only from the k calibration trials** (k images shown in a calibration session; labels are free because we know what was shown). Optionally, the unlabelled trials recorded during use (the 200 test trials) are added — those rows are marked *transductive*.

| Method | What it uses | Role |
|---|---|---|
| `direct` | nothing | baseline |
| `MM_k` | mean/std of the k calibration embeddings, shrunk towards the source-subject statistics (n0 = 50) | baseline (fair version of 08) |
| `ortho_k`, `ridge_k` | MM_k + map fitted on the k pairs (rho 0.1 / lam 1, as in 08) | baselines |
| **`SHRINK` (proposed)** | MM_k + orthogonal map with **k-adaptive pull to identity** rho_k = 0.1·(1 + K0/k) and output blend alpha_k = k/(k+K0), K0 = 100 (fixed in advance) | candidate method: never worse than k = 0, improves smoothly |
| `SHRINK + L2-SP` | SHRINK after fine-tuning the EEG encoder with a penalty to the source weights | exploratory |
| `TTA-MM` | moment matching on the 200 unlabelled usage trials | transductive reference (≈ 41.9 in 08A) |
| **`SHRINK + TTA`** | statistics from k calibration + 200 usage trials, plus 64 pseudo-landmarks with margin weights next to the k labelled pairs | transductive candidate |
| sensitivity | SHRINK with K0 ∈ {25, 400}, n0 ∈ {10, 200} | robustness of the fixed choice |

k = 0, 10, 25, 50, 100, 250, 500, 1000 (≈ 0 … 13 min), 3 random draws, 10 LOSO folds. Main hyper-parameters are fixed before seeing test results.

**Drive is full:** this notebook only READS from Drive. Results are written to the runtime disk and downloaded as a zip at the end. Missing LOSO models are retrained (~1 min each on L4) and kept on the runtime disk only. **GPU:** L4.

In [ ]:
# 0. Setup
import os, sys, gc, json, math, time, random, shutil, glob, copy, importlib.util, subprocess
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
DRIVE  = os.environ.get("EEG_DRIVE", "/content/drive/MyDrive/eeg_results")
DCACHE = f"{DRIVE}/cache"
ROOT   = os.environ.get("EEG_ROOT", "/content/EEG_Data")
LOCAL  = os.environ.get("EEG_LOCAL", "/content/cache")
OUT    = "/content/v9_local"   # runtime disk (Drive is full); zipped and downloaded at the end
V6, V6B = f"{DRIVE}/v6", f"{DRIVE}/v6b"
for d in [DRIVE, DCACHE, ROOT, LOCAL, OUT]: os.makedirs(d, exist_ok=True)
ALL_SUBJECTS = [f"sub-{i:02d}" for i in range(1, 11)]
SMOKE = os.environ.get("EEG_SMOKE") == "1"           # tiny synthetic run for code checking only

import numpy as np
for pkg, pip_name in [("huggingface_hub", "huggingface_hub"), ("pandas", "pandas"), ("scipy", "scipy")]:
    if not SMOKE and importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
device = "cuda" if torch.cuda.is_available() else "cpu"
AMP = device == "cuda" and torch.cuda.get_device_capability()[0] >= 8      # bfloat16 needs Ampere or newer (not T4)
print("device:", device, torch.cuda.get_device_name(0) if device == "cuda" else "", "| bf16:", AMP, "| torch", torch.__version__)
print(f"Drive free: {shutil.disk_usage(DRIVE).free / 1e9:.1f} GB (features need ~1.2 GB)")

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def local_copy(name):
    dst = f"{LOCAL}/{name}"
    if not os.path.exists(dst): shutil.copy(f"{DCACHE}/{name}", dst + ".tmp"); os.replace(dst + ".tmp", dst)
    return dst

def save_json(obj, path):
    json.dump(obj, open(path + ".tmp", "w")); os.replace(path + ".tmp", path)

def free_gpu():
    gc.collect()
    if device == "cuda": torch.cuda.empty_cache()

In [ ]:
# 1. Image list (same order as notebooks 04/05) and the validation split
META = f"{DCACHE}/meta_v6.pt"
if not os.path.exists(META):
    import tarfile
    from huggingface_hub import hf_hub_download
    if not os.path.exists(f"{ROOT}/images_set/image_metadata.npy"):
        tar_path = hf_hub_download(repo_id="LidongYang/EEG_Image_decode", filename="images_set.tar.gz", repo_type="dataset", local_dir=ROOT)
        with tarfile.open(tar_path, "r:gz") as t: t.extractall(path=ROOT)
    obj = np.load(f"{ROOT}/images_set/image_metadata.npy", allow_pickle=True)
    meta = obj.item() if isinstance(obj, np.ndarray) and obj.dtype == object and obj.ndim == 0 else obj
    train_files, test_files = list(meta["train_img_files"]), list(meta["test_img_files"])
    def build_index(folder):
        idx = {}
        for dp, _, fs in os.walk(folder):
            for f in fs: idx.setdefault(f, os.path.join(dp, f))
        return idx
    img_root = f"{ROOT}/images_set"
    idx_train = build_index(f"{img_root}/training_images")
    idx_test = build_index(f"{img_root}/test_images") if os.path.isdir(f"{img_root}/test_images") else build_index(img_root)
    torch.save({"train_files": train_files, "test_files": test_files,
                "train_rel": [os.path.relpath(idx_train[f], ROOT) for f in train_files],
                "test_rel": [os.path.relpath(idx_test[f], ROOT) for f in test_files],
                "n_concepts": len(set(meta["train_img_concepts"]))}, META)
M = torch.load(META, weights_only=False)
N_TR, N_TE = len(M["train_files"]), len(M["test_files"])
PER = N_TR // M["n_concepts"]
IS_VAL = (np.arange(N_TR) % PER) == (PER - 1)              # last image of every training concept (same as 01–05)
print(f"train images {N_TR} | test images {N_TE} | concepts {M['n_concepts']} | val images {IS_VAL.sum()}")

In [ ]:
# 2. Cached image features of both backbones (written by notebooks 06 / 06b) and z-scored targets (training statistics)
def load_feats(b):
    p = f"{DCACHE}/feat6_{b}.pt"
    if os.path.exists(p): return torch.load(p, weights_only=False)
    parts = f"{DCACHE}/feat6_{b}_parts"                     # 06b saved chunks; the merged file may not have synced
    assert os.path.isdir(parts), f"neither {p} nor {parts} found"
    def merge(tag):
        files = sorted(glob.glob(f"{parts}/{tag}_*.pt")); print(f"  {b} {tag}: {len(files)} chunks")
        ch = [torch.load(f) for f in files]
        return {k: torch.cat([c[k] for c in ch]) for k in ch[0]}
    tr, te = merge("train"), merge("test")
    layers = sorted({l for l, _ in tr})
    assert len(tr[(layers[0], "mean")]) == N_TR and len(te[(layers[0], "mean")]) == N_TE, "chunks incomplete"
    return {"layers": layers, "tr": tr, "te": te}

FE = {b: load_feats(b) for b in ["internvit6b", "dinov2g"]}
for b, d in FE.items(): print(b, "layers", d["layers"], "dim", d["tr"][(d["layers"][0], "mean")].shape[1])

def target_te(b, layer):
    tr, te = FE[b]["tr"][(layer, "mean")].float(), FE[b]["te"][(layer, "mean")].float()
    m, s = tr.mean(0, keepdim=True), tr.std(0, keepdim=True) + 1e-4
    return ((te - m) / s).to(device)

CH63 = ["Fp1","Fp2","AF7","AF3","AFz","AF4","AF8","F7","F5","F3","F1","F2","F4","F6","F8","FT9","FT7","FC5","FC3","FC1",
        "FCz","FC2","FC4","FC6","FT8","FT10","T7","C5","C3","C1","Cz","C2","C4","C6","T8","TP9","TP7","CP5","CP3","CP1",
        "CPz","CP2","CP4","CP6","TP8","TP10","P7","P5","P3","P1","Pz","P2","P4","P6","P8","PO7","PO3","POz","PO4","PO8",
        "O1","Oz","O2"]
_, N_CH_ALL, N_T = np.load(f"{DCACHE}/sub-01_test_z16.npy", mmap_mode="r").shape
def load_test(sub):
    x = np.load(local_copy(f"{sub}_test_z16.npy"), mmap_mode="r")
    return torch.from_numpy(np.array(x)).to(device)
def load_train(sub):
    return torch.from_numpy(np.array(np.load(local_copy(f"{sub}_train_z16.npy"), mmap_mode="r"))).to(device)
def target_tr(b, layer):
    tr = FE[b]["tr"][(layer, "mean")].float()
    m, s = tr.mean(0, keepdim=True), tr.std(0, keepdim=True) + 1e-4
    return ((tr - m) / s).to(device)
VAL = torch.from_numpy(np.where(IS_VAL)[0]).to(device)


In [ ]:
# 4. Model: EEGProject (MLP with residual bottleneck, as in Shallow Alignment) + linear image projector(s)
#    Single-layer target, or a learned softmax routing over several layers (global SAMGA-style prior, no subject residual).
EMB = 1024

class EEGProject(nn.Module):
    def __init__(self, n_in, d=EMB, drop=0.3):
        super().__init__()
        self.inp = nn.Linear(n_in, d)
        self.res = nn.Sequential(nn.GELU(), nn.Linear(d, d), nn.Dropout(drop))
        self.norm = nn.LayerNorm(d)
    def forward(self, x):
        h = self.inp(x.flatten(1)); return self.norm(h + self.res(h))

class ImageSide(nn.Module):
    def __init__(self, dims, init_logits=None):
        super().__init__()
        self.proj = nn.ModuleList([nn.Linear(d, EMB) for d in dims])
        self.logits = nn.Parameter(torch.tensor(init_logits, dtype=torch.float) if init_logits is not None else torch.zeros(len(dims)),
                                   requires_grad=len(dims) > 1)
    def weights(self): return F.softmax(self.logits, 0)
    def forward(self, feats):                   # feats: list of (N, d_l), same rows
        w = self.weights()
        return sum(w[i] * p(f) for i, (p, f) in enumerate(zip(self.proj, feats)))

class Aligner(nn.Module):
    def __init__(self, n_in, dims, init_logits=None):
        super().__init__()
        self.eeg, self.img = EEGProject(n_in), ImageSide(dims, init_logits)
        self.log_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))
    def scale(self): return self.log_scale.exp().clamp(max=100)

def clip_loss(ze, zi, scale):
    ze, zi = F.normalize(ze, dim=-1), F.normalize(zi, dim=-1)
    logits = scale * ze @ zi.T; y = torch.arange(len(ze), device=ze.device)
    return (F.cross_entropy(logits, y) + F.cross_entropy(logits.T, y)) / 2

@torch.no_grad()
def embed_eeg(model, X, bs=4096):
    model.eval(); out = []
    for i in range(0, len(X), bs):
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=AMP):
            out.append(F.normalize(model.eeg(X[i:i + bs].float()).float(), dim=-1))
    return torch.cat(out)

@torch.no_grad()
def embed_img(model, feats):
    model.eval(); return F.normalize(model.img(feats).float(), dim=-1)

@torch.no_grad()
def metrics(sim):
    # sim: queries x candidates, the correct candidate of query i is candidate i
    r = (sim > sim.diag().unsqueeze(1)).sum(1)
    return {"top1": (r < 1).float().mean().item() * 100, "top5": (r < 5).float().mean().item() * 100}

EPOCHS, BATCH, LR, WD, EVAL_EVERY = (60, 1024, 1e-4, 1e-4, 5) if not SMOKE else (3, 64, 1e-3, 1e-4, 1)
ROUTE_INIT = [-2.0, -1.0, 0.0, -1.0, -2.0]

def train(X, img_of, T_tr, val_sets, seed=0, log=None, init_logits=None):
    # X: EEG trials, img_of[i]: training image of trial i, T_tr: list of target matrices (N_TR x d_l)
    # val_sets: list of (EEG rows, image rows) used for model selection (held-out training images)
    set_seed(seed)
    model = Aligner(X[0].numel(), [t.shape[1] for t in T_tr], init_logits).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    tr_rows = torch.where(~torch.from_numpy(IS_VAL).to(device)[img_of])[0]
    steps = EPOCHS * math.ceil(len(tr_rows) / BATCH)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=steps, pct_start=0.05)
    best, best_state, best_ep = -1, None, 0
    for ep in range(1, EPOCHS + 1):
        model.train(); perm = tr_rows[torch.randperm(len(tr_rows), device=device)]
        for i in range(0, len(perm), BATCH):
            b = perm[i:i + BATCH]
            if len(b) < 2: continue
            imgs = img_of[b]
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=AMP):
                ze = model.eeg(X[b].float()); zi = model.img([t[imgs] for t in T_tr])
            loss = clip_loss(ze.float(), zi.float(), model.scale())
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sched.step()
        if ep % EVAL_EVERY == 0 or ep == EPOCHS:
            zi_val = embed_img(model, [t[VAL] for t in T_tr])
            va = float(np.mean([metrics(embed_eeg(model, X[er]) @ zi_val[ir].T)["top1"] for er, ir in val_sets]))
            if log: log(ep, va, model)
            if va > best: best, best_ep, best_state = va, ep, copy.deepcopy(model.state_dict())
    model.load_state_dict(best_state)
    return model, best_ep, best
@torch.no_grad()
def metrics(sim):
    r = (sim > sim.diag().unsqueeze(1)).sum(1)
    return {"top1": (r < 1).float().mean().item() * 100, "top5": (r < 5).float().mean().item() * 100}

AMP = AMP if "AMP" in globals() else False
# the four LOSO models of every fold: (name, folder, file tag, backbone, layers)
MODELS = [("intern_route", V6B, "route_c", "internvit6b", [20, 24, 28, 32, 36]),
          ("intern_L28",   V6B, "L28",     "internvit6b", [28]),
          ("dino_route",   V6,  "route",   "dinov2g",     [20, 24, 28, 32, 36]),
          ("dino_L20",     V6,  "L20",     "dinov2g",     [20])]
PRIMARY = "intern_route"

MODEL_LOCAL = f"{LOCAL}/models"; os.makedirs(MODEL_LOCAL, exist_ok=True)
ROUTE_INIT5 = [-2.0, -1.0, 0.0, -1.0, -2.0]

def model_path(folder, tag, sub):
    loc = f"{MODEL_LOCAL}/{os.path.basename(folder)}_loso_{tag}_{sub}.pt"
    return loc if os.path.exists(loc) else f"{folder}/loso_{tag}_{sub}.pt"

def retrain(folder, tag, sub, backbone, layers):
    """same recipe as notebooks 06/06b cell 6 (9 source subjects, 63 ch, seed 0, best epoch on source validation images)"""
    t0 = time.time(); srcs = [s for s in (ALL_SUBJECTS if not SMOKE else ALL_SUBJECTS[:3]) if s != sub]
    X = torch.empty((len(srcs) * N_TR, N_CH_ALL, N_T), dtype=torch.float16, device=device)
    for k, s_ in enumerate(srcs): X[k * N_TR:(k + 1) * N_TR] = load_train(s_)
    img_of = torch.arange(len(X), device=device) % N_TR
    vsets = [(k * N_TR + VAL, torch.arange(len(VAL), device=device)) for k in range(min(3, len(srcs)))]
    T_tr = [target_tr(backbone, l) for l in layers]
    m, ep, va = train(X, img_of, T_tr, vsets, 0, None, ROUTE_INIT5[-len(layers):] if len(layers) > 1 else None)
    del X, T_tr; free_gpu()
    loc = f"{MODEL_LOCAL}/{os.path.basename(folder)}_loso_{tag}_{sub}.pt"
    torch.save(m.state_dict(), loc)
    # Drive is full: keep the retrained model on the runtime disk only
    print(f"  retrained {tag} {sub}: best epoch {ep}, val {va:.1f} ({(time.time()-t0)/60:.1f} min)", flush=True)

def load_model(folder, tag, sub, backbone, layers):
    if not os.path.exists(model_path(folder, tag, sub)): retrain(folder, tag, sub, backbone, layers)
    dims = [FE[backbone]["tr"][(l, "mean")].shape[1] for l in layers]
    m = Aligner(N_CH_ALL * N_T, dims).to(device)
    m.load_state_dict(torch.load(model_path(folder, tag, sub), map_location=device)); m.eval()
    return m

missing = [(n, sub) for n, folder, tag, bb, layers in MODELS for sub in (ALL_SUBJECTS if not SMOKE else ALL_SUBJECTS[:2])
           if not os.path.exists(model_path(folder, tag, sub))]
print(f"{len(missing)} LOSO model files missing -> retrained in cell 4 (~1 min each on A100/L4, ~4 min on T4):", missing)


In [ ]:
# 3. Maps, scoring and embeddings
CSLS_K, RHO = 10, 0.1
from scipy.stats import wilcoxon

@torch.no_grad()
def csls(S, k=CSLS_K):
    rz = S.topk(min(k, S.shape[1]), dim=1).values.mean(1, keepdim=True)
    rt = S.topk(min(k, S.shape[0]), dim=0).values.mean(0, keepdim=True)
    return 2 * S - rz - rt

def mm(Z, ref_Z, ref_T):
    """moment matching: standardise Z with the statistics of ref_Z, rescale to the statistics of ref_T"""
    return F.normalize((Z - ref_Z.mean(0)) / (ref_Z.std(0) + 1e-6) * ref_T.std(0) + ref_T.mean(0), dim=-1)

def mnn_pairs(S):
    top2 = S.topk(2, dim=1); j = top2.indices[:, 0]; i = torch.arange(len(S), device=S.device)
    mnn = S.argmax(0)[j] == i
    margin = (top2.values[:, 0] - top2.values[:, 1]).clamp(min=1e-4)
    q = torch.where(mnn)[0]
    return q, j[q], margin[q]

def top_by(q, c, key, n):
    o = key.argsort(descending=True)[:n]; return q[o], c[o], key[o]

@torch.no_grad()
def fit_ortho(Xl, Yl, w=None, rho=RHO):
    w = torch.ones(len(Xl), device=Xl.device) if w is None else len(Xl) * w / w.sum()
    wn = w / w.sum(); muX = (wn[:, None] * Xl).sum(0); muY = (wn[:, None] * Yl).sum(0)
    C = ((Xl - muX) * w[:, None]).T @ (Yl - muY)
    Mx = C + rho * torch.linalg.matrix_norm(C, ord=2) * torch.eye(C.shape[0], device=C.device)
    U, _, Vh = torch.linalg.svd(Mx.double()); R = (U @ Vh).float()
    return lambda Z: F.normalize((Z - muX) @ R + muY, dim=-1)

@torch.no_grad()
def fit_ridge(Xl, Yl, lam_rel=1.0):
    """affine map pulled towards the identity: W = argmin |Xc W - Yc|^2 + lam |W - I|^2, lam = lam_rel * mean eigenvalue"""
    muX, muY = Xl.mean(0), Yl.mean(0); Xc, Yc = (Xl - muX).double(), (Yl - muY).double()
    d = Xc.shape[1]; G = Xc.T @ Xc; lam = lam_rel * torch.trace(G) / d + 1e-6
    I = torch.eye(d, device=Xl.device, dtype=torch.double)
    W = torch.linalg.solve(G + lam * I, Xc.T @ Yc + lam * I).float()
    return lambda Z: F.normalize((Z - muX) @ W + muY, dim=-1)

def rank_metrics(S):
    return metrics(S)

# z-scoring statistics of every (backbone, layer) on the training images, kept on CPU; targets built in chunks
STATS = {}
def stat(b, l):
    if (b, l) not in STATS:
        tr = FE[b]["tr"][(l, "mean")].float(); STATS[(b, l)] = (tr.mean(0, keepdim=True), tr.std(0, keepdim=True) + 1e-4)
    return STATS[(b, l)]

@torch.no_grad()
def embed_train_images(m, b, layers, rows=None, bs=2048):
    rows = torch.arange(N_TR) if rows is None else rows.cpu()
    out = []
    for i in range(0, len(rows), bs):
        r = rows[i:i + bs]
        feats = [((FE[b]["tr"][(l, "mean")][r].float() - stat(b, l)[0]) / stat(b, l)[1]).to(device) for l in layers]
        out.append(embed_img(m, feats))
    return torch.cat(out)


In [ ]:
# 4. Calibration methods
RHO0, K0, N0 = 0.1, 100, 50                     # fixed in advance
from scipy.stats import wilcoxon

def recover(Zm, T, n=64, rho=RHO0):
    S = csls(Zm @ T.T); q, c, mg = mnn_pairs(S)
    if len(q) < 3: return S
    q, c, mg = top_by(q, c, mg, n)
    return csls(fit_ortho(Zm[q], T[c], mg, rho)(Zm) @ T.T)

def shrunk_stats(Zk, mu_src, sd_src, n0=N0):
    """mean/std of the new user's embeddings from k trials, shrunk towards the source-subject statistics"""
    k = len(Zk)
    if k == 0: return mu_src, sd_src
    mu_k = Zk.mean(0); var_k = Zk.var(0, unbiased=False) if k > 1 else torch.zeros_like(mu_k)
    mu = (k * mu_k + n0 * mu_src) / (k + n0)
    var = (k * (var_k + (mu_k - mu) ** 2) + n0 * (sd_src ** 2 + (mu_src - mu) ** 2)) / (k + n0)
    return mu, var.sqrt()

def mm_stats(Z, mu, sd, T_ref):
    return F.normalize((Z - mu) / (sd + 1e-6) * T_ref.std(0) + T_ref.mean(0), dim=-1)

def shrink_map(Xl, Yl, k, K0=K0, w=None):
    """orthogonal map with a k-adaptive pull to the identity + output blend; identity when k = 0"""
    if k == 0: return lambda Z: Z
    f = fit_ortho(Xl, Yl, w, rho=RHO0 * (1 + K0 / k)); a = k / (k + K0)
    return lambda Z: F.normalize((1 - a) * Z + a * f(Z), dim=-1)

def finetune_l2sp(m, Xk, Tk, T_neg, seed, lam=10.0, epochs=40, lr=1e-4):
    """fine-tune the EEG encoder on k pairs with a relative L2 penalty towards the source weights (L2-SP)"""
    set_seed(seed); mf = copy.deepcopy(m); mf.train()
    for p in mf.img.parameters(): p.requires_grad_(False)
    ref = [p.detach().clone() for p in mf.eeg.parameters()]
    norm0 = sum((r ** 2).sum() for r in ref)
    opt = torch.optim.AdamW(mf.eeg.parameters(), lr=lr, weight_decay=0)
    cand = torch.cat([Tk, T_neg]); bs = min(len(Xk), 128)
    for ep in range(epochs):
        perm = torch.randperm(len(Xk), device=device)
        for i in range(0, len(Xk), bs):
            b = perm[i:i + bs]
            ze = F.normalize(mf.eeg(Xk[b].float()), dim=-1)
            pen = sum(((p - r) ** 2).sum() for p, r in zip(mf.eeg.parameters(), ref)) / norm0
            loss = F.cross_entropy(mf.scale() * ze @ cand.T, b) + lam * pen
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    mf.eval(); return mf


In [ ]:
# 5. Run: 10 LOSO folds x k x draws
FOLDS = ALL_SUBJECTS if not SMOKE else ALL_SUBJECTS[:2]
KS = [0, 10, 25, 50, 100, 250, 500, 1000] if not SMOKE else [0, 5, 10]
DRAWS = 3 if not SMOKE else 1
N_SRC = 2000 if not SMOKE else 30               # source trials per source subject for the source statistics
rows = []
name, folder, tag, bb, layers = MODELS[0]       # route_c, InternViT-6B

for sub in FOLDS:
    t0 = time.time()
    m = load_model(folder, tag, sub, bb, layers)
    Xt, Xs = load_test(sub), load_train(sub)
    T = embed_img(m, [target_te(bb, l) for l in layers]); Ttr = embed_train_images(m, bb, layers)
    Zt, Zs = embed_eeg(m, Xt), embed_eeg(m, Xs)
    g = torch.Generator(device="cpu").manual_seed(0)
    Zsrc = torch.cat([embed_eeg(m, load_train(s)[torch.randperm(N_TR, generator=g)[:N_SRC].to(device)])
                      for s in ALL_SUBJECTS if s != sub and (not SMOKE or s in ALL_SUBJECTS[:3])])
    mu_src, sd_src = Zsrc.mean(0), Zsrc.std(0)
    def add(method, k, d, S, transductive=False):
        mt = metrics(S); rows.append(dict(subject=sub, method=method, k=k, draw=d, transductive=transductive,
                                          top1=mt["top1"], top5=mt["top5"]))
    add("direct", 0, 0, Zt @ T.T)
    add("TTA-MM (usage trials)", 0, 0, csls(mm(Zt, Zt, T) @ T.T), True)
    add("08 full-session MM (3.7 h, reference)", 0, 0, csls(mm(Zt, Zs, Ttr) @ T.T))
    gk = torch.Generator(device="cpu").manual_seed(1)
    for k in KS:
        for d in (range(DRAWS) if k > 0 else [0]):
            idx = torch.randperm(N_TR, generator=gk)[:k].to(device)
            Zk, Tk = Zs[idx], Ttr[idx]
            mu, sd = shrunk_stats(Zk, mu_src, sd_src)
            Zt_k, Zk_k = mm_stats(Zt, mu, sd, Ttr), mm_stats(Zk, mu, sd, Ttr)
            add("MM_k", k, d, csls(Zt_k @ T.T))
            add("SHRINK (proposed)", k, d, csls(shrink_map(Zk_k, Tk, k)(Zt_k) @ T.T))
            for K0s in [25, 400]: add(f"SHRINK K0={K0s}", k, d, csls(shrink_map(Zk_k, Tk, k, K0=K0s)(Zt_k) @ T.T))
            for n0 in [10, 200]:
                mu2, sd2 = shrunk_stats(Zk, mu_src, sd_src, n0)
                add(f"SHRINK n0={n0}", k, d, csls(shrink_map(mm_stats(Zk, mu2, sd2, Ttr), Tk, k)(mm_stats(Zt, mu2, sd2, Ttr)) @ T.T))
            if k > 0:
                add("ortho_k", k, d, csls(fit_ortho(Zk_k, Tk)(Zt_k) @ T.T))
                add("ridge_k", k, d, csls(fit_ridge(Zk_k, Tk, 1.0)(Zt_k) @ T.T))
            # transductive: statistics from k calibration + 200 usage trials, k labelled pairs + 64 pseudo-landmarks
            Zu = torch.cat([Zk, Zt]); mu_u, sd_u = shrunk_stats(Zu, mu_src, sd_src)
            Zt_u = mm_stats(Zt, mu_u, sd_u, Ttr); Zk_u = mm_stats(Zk, mu_u, sd_u, Ttr)
            S0 = csls(Zt_u @ T.T); q, c, mg = mnn_pairs(S0); q, c, mg = top_by(q, c, mg, 64)
            Xl = torch.cat([Zk_u, Zt_u[q]]); Yl = torch.cat([Tk, T[c]])
            w = torch.cat([torch.full((k,), float(mg.mean()) * 2, device=device), mg])
            n_eff = k + len(q)
            add("SHRINK + TTA (proposed, transductive)", k, d, csls(shrink_map(Xl, Yl, n_eff, w=w)(Zt_u) @ T.T), True)
            if k >= 25:
                neg = torch.randperm(N_TR, generator=gk)[:1000].to(device)
                mf = finetune_l2sp(m, Xs[idx], Tk, Ttr[neg], seed=d)
                Zt_f, Zk_f = embed_eeg(mf, Xt), embed_eeg(mf, Xs[idx])
                muf, sdf = shrunk_stats(Zk_f, mu_src, sd_src)
                add("SHRINK + L2-SP fine-tune", k, d,
                    csls(shrink_map(mm_stats(Zk_f, muf, sdf, Ttr), Tk, k)(mm_stats(Zt_f, muf, sdf, Ttr)) @ T.T))
                del mf; free_gpu()
    R = pd.DataFrame([r for r in rows if r["subject"] == sub]).groupby(["method", "k"]).top1.mean()
    print(f"{sub} ({(time.time()-t0)/60:.1f} min): direct {R[('direct', 0)]:.1f} | MM_k k=25 {R[('MM_k', 25)] if ('MM_k', 25) in R else float('nan'):.1f} "
          f"| ortho k=25 {R.get(('ortho_k', 25), float('nan')):.1f} | SHRINK k=25 {R.get(('SHRINK (proposed)', 25), float('nan')):.1f} "
          f"| SHRINK k=250 {R.get(('SHRINK (proposed)', 250), float('nan')):.1f} | +TTA k=25 {R.get(('SHRINK + TTA (proposed, transductive)', 25), float('nan')):.1f}", flush=True)
    del m, Xt, Xs, Ttr, Zs, Zsrc; free_gpu()

D9 = pd.DataFrame(rows); D9.to_csv(f"{OUT}/calibration_v9.csv", index=False)


In [ ]:
# 6. Summary: Top-1 vs calibration minutes, never-worse check, significance, download
CURVE = D9.groupby(["method", "k"]).top1.mean().unstack("k").round(1)
order = ["direct", "08 full-session MM (3.7 h, reference)", "TTA-MM (usage trials)", "MM_k", "ortho_k", "ridge_k",
         "SHRINK (proposed)", "SHRINK + L2-SP fine-tune", "SHRINK + TTA (proposed, transductive)",
         "SHRINK K0=25", "SHRINK K0=400", "SHRINK n0=10", "SHRINK n0=200"]
print("Top-1 on the unseen subject (mean over 10 subjects x draws); columns = k calibration images")
print("minutes:", {k: round(k * 4 * 0.2 / 60, 1) for k in KS})
print(CURVE.loc[[o for o in order if o in CURVE.index]].to_string())

# never-worse: per subject, mean over draws, compared with the same method at k = 0 (and with MM_k at the same k)
per = D9.groupby(["method", "subject", "k"]).top1.mean()
print("\nPer-subject check (10 subjects): how often is the method WORSE than its own k = 0, and vs ortho_k / ridge_k")
for meth in ["ortho_k", "ridge_k", "SHRINK (proposed)", "SHRINK + TTA (proposed, transductive)"]:
    line = []
    for k in [k for k in KS if k > 0]:
        try:
            base0 = per.xs(("MM_k", 0), level=("method", "k")) if "TTA" not in meth else per.xs(("SHRINK + TTA (proposed, transductive)", 0), level=("method", "k"))
            cur = per.xs((meth, k), level=("method", "k"))
            line.append(f"k={k}: worse {int((cur < base0 - 1e-9).sum())}/10")
        except KeyError: pass
    print(f"  {meth:40s} " + " | ".join(line))

print("\nSHRINK vs best baseline (max of ortho_k, ridge_k, MM_k) per k: mean diff, wins/10, Wilcoxon p")
for k in [k for k in KS if k > 0]:
    s = per.xs(("SHRINK (proposed)", k), level=("method", "k"))
    b = pd.concat([per.xs((mm_, k), level=("method", "k")) for mm_ in ["ortho_k", "ridge_k", "MM_k"]], axis=1).max(axis=1)
    d = s - b; p = wilcoxon(d).pvalue if (d != 0).any() else float("nan")
    print(f"  k={k:5d}: {d.mean():+5.1f}  wins {int((d > 0).sum())}/10  p={p:.3f}   (SHRINK {s.mean():.1f} vs best baseline {b.mean():.1f})")

CURVE.to_csv(f"{OUT}/calibration_curve_v9.csv")
import shutil
shutil.make_archive("/content/v9_results", "zip", OUT)
try:
    from google.colab import files; files.download("/content/v9_results.zip")
except Exception as e: print("download skipped:", e)
